## Imports 📥
 ---

In [3]:
import csv
import os
from typing import List

In [4]:
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_text_splitters import RecursiveCharacterTextSplitter

d:\Aditya\Agentic AI\Amazon_Bedrock_Customer_Care_Agent\Customer_Care_Agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain.agents import create_agent
from dotenv import load_dotenv

load_dotenv()

C:\Users\hp\AppData\Local\Temp\ipykernel_15196\2950575863.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


True

## Making Vector Database ⚙️🛢
---

### Loading Document

In [ ]:
def load_csv(path:str)->List[Document]:
    docs = []
    with open(path,"r",encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            q = row["question"].strip()
            a = row["answer"].strip()
            docs.append(Document(page_content=f"Q: {q}\nA: {a}"))
    return docs

### loading Embedding model

In [ ]:
docs = load_csv("./lauki_qna.csv")

In [8]:
emb = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 730.99it/s]


### Splitting and storing data in Data base FAISS

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
chunks = splitter.split_documents(docs)
store = FAISS.from_documents(chunks, emb)

In [ ]:
store.save_local("faiss_index") 

 ---
 ---
# RAG 📚 ---> ⚙️ ---->  𐙚‧₊˚📜✩ ₊˚⊹♡ 

## 1. Loading Vector Database 🛢
 ---

In [ ]:
from langchain_community.vectorstores import FAISS

In [ ]:
emb = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
)

In [9]:
store = FAISS.load_local(
    "faiss_index",
    emb,
    allow_dangerous_deserialization=True
    )

## 2. Retriving 🛢🔀📃
 ---

### Making Semantic_Search Tool

In [ ]:
@tool
def search(query:str)->str:
    """Search the FAQ knowledge base for relevant information.
        Use this tool when the user asks questions about products, services, or policies.
        
        Args:
            query: The search query to find relevant FAQ entries
            
        Returns:
            Relevant FAQ entries that might answer the question
        """

    


    context = "\n\n---\n\n".join([
        f"FAQ Entry {i+1}:\n{doc.page_content}" 
        for i, doc in enumerate(results)
    ])

    return f"Found {len(results)} relevant FAQ entries:\n\n{context}"


### Initialising Model, Tool, and System_prompt

In [12]:
model = ChatGroq(
    model ="openai/gpt-oss-20b",
    temperature=0.7
)

In [11]:
tools = [search]

In [13]:
system_prompt = """You are a helpful FAQ assistant with access to a knowledge base.

Your goal is to answer user questions accurately using the available tools.

Guidelines:
1. Start by using the search_faq tool to find relevant information
2. If the initial search doesn't provide enough info, use search_detailed_faq for more results
3. If the query is complex, use reformulate_query to search different aspects
4. Synthesize information from multiple tool calls if needed
5. Always provide a clear, concise answer based on the retrieved information
6. If you cannot find relevant information, clearly state that

Think step-by-step and use tools strategically to provide the best answer.

and always say thankyou for your query at start """

## 3. Agent 🤖

In [14]:
agent = create_agent(
    model = model,
    tools = tools,
    system_prompt = system_prompt
)

In [16]:
result = agent.invoke({"messages": [("human", "I deactivate an addon?")]})
print(result['messages'][-1].content)

Thank you for your query!  

According to our FAQ, **addons cannot be manually deactivated before their scheduled expiry**.  
- They will automatically deactivate when their validity period ends.  
- Mid‑cycle deactivation is not supported to keep pricing fair.  

If you have an unused addon and believe you should receive a prorated refund (e.g., due to an app glitch), please **contact support**. They can review your case and issue a refund if eligible. Otherwise, simply let the addon expire and monitor usage to ensure you get the most value.  

If you need further assistance, feel free to let us know!
